# 9b · Grounding — run the judge over a split, then prune

Every assertion in every graph of a split is put to a judge with the **full article** in front of it:
*does this text support this assertion?* What survives is written out as a **pruned graph** — the
artefact downstream evaluation (nb10 / MINE) should consume, since every assertion left in it has
been checked against its source.

**The judge itself lives in `grounding_core.py`, not in this notebook.** Its twin, `9d_judge_benchmark`,
measures that same judge against 500 gold-labelled assertions. A prompt edited here is measured there;
if the two had their own copies, the benchmark would stop describing this run.

**Read the judge's number with its known error in mind.** On the 2026-08-19 benchmark, qwen3-32b-think
recovered 75% of broken assertions but only 23–41% of the *real* extractor errors in the sample — and
it over-reported the grounding rate on real graph rows by ~4 points (95.2% judged vs 91.2% human).
The rate this notebook prints is a **ceiling**, and the gap lives in wrong properties, wrong types and
reversed arrows. Run 9d after any prompt change rather than assuming it improved things.

| output | what it is |
|---|---|
| `nary_graph_grounded_fulltext.json` | the input graph with a `grounding` block on every node and relation |
| `nary_graph_grounded_pruned.json` | **the same graph with everything ungrounded removed** — for nb10 |
| `grounding_fulltext_judgments.json` / `.csv` | one flat row per decision, for review |
| `_global/grounding_summary_<tag>.json` | corpus-level rates + what pruning removed |

In [1]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [2]:
# ---- WHICH GRAPHS ---------------------------------------------------------------------------
# Each knob below falls back to $GROUND_* so a headless batch can sweep models/tags without
# editing this notebook. Unset means the literal default here, so an interactive run is unchanged.
import os as _os
RESULTS_MODEL = _os.environ.get("GROUND_MODEL") or "gemini-2.5-flash"   # whose graphs: results/<RESULTS_MODEL>/

# ---- WHICH JUDGE ----------------------------------------------------------------------------
# One judge for every model's graphs is what makes the arms comparable — but it must not be the
# model that WROTE the graphs, or the grounding rate is a self-report. 9d ranks the candidates.
BACKEND = _os.environ.get("GROUND_BACKEND") or "qwen3-32b-think"   # needs the vLLM tunnel up

# ---- WHICH ARTICLES -------------------------------------------------------------------------
#   None            -> every article under results/<RESULTS_MODEL>/ that has GRAPH_NAME
#   ["art5", ...]   -> exactly these (ids without the graph are reported and skipped)
ARTICLES = None

# --- Config ----------------------------------------------------------------------------------
from pathlib import Path

GRAPH_NAME  = _os.environ.get("GROUND_GRAPH") or "nary_graph_dedup_main_val_r1_extended_consolidated.json"
OUT_NAME    = "nary_graph_grounded_fulltext.json"                        # judged, written beside it
PRUNED_NAME = "nary_graph_grounded_pruned.json"                          # judged AND pruned -> nb10
TRACK_DIR   = _os.environ.get("GROUND_TRACK") or "e2e"
RUN_TAG     = _os.environ.get("GROUND_TAG") or "main_val_r1"   # names the _global summary

# Judge knobs -> pushed into grounding_core in the client cell below.
JUDGE_BATCH = 20 if "think" in BACKEND else 60   # a thinking judge reasons over the whole batch
JUDGE_RETRIES = 3
JUDGE_RETRY_BACKOFF = 5
WORKERS = 4                          # ARTICLES in flight at once; 1 while debugging

# Pruning. Unjudged items (a judge call that failed, not a fact that is wrong) are KEPT by default
# so API flakiness cannot masquerade as a worse extractor; they are counted either way.
DROP_UNJUDGED = False
DROP_ORPHAN_ENTITIES = True          # an entity no surviving relation mentions is not still asserted

RESULTS_DIR = ROOT / "results" / RESULTS_MODEL
ARTICLE_DIR = ROOT / "dataset" / "articles"
GLOBAL_DIR  = RESULTS_DIR / "_global"

import re as _re
def _natkey(s):
    return [int(t) if t.isdigit() else t for t in _re.split(r"(\d+)", str(s))]

def resolve_articles(ids=None):
    has_graph = lambda a: (RESULTS_DIR / a / TRACK_DIR / GRAPH_NAME).exists()
    if ids is None:
        return sorted((p.name for p in RESULTS_DIR.iterdir() if p.is_dir() and has_graph(p.name)),
                      key=_natkey)
    missing = [a for a in ids if not has_graph(a)]
    if missing:
        print(f"!! skipped — no {TRACK_DIR}/{GRAPH_NAME}: {missing}")
    return [a for a in ids if has_graph(a)]

ARTICLE_IDS = resolve_articles(ARTICLES)
assert ARTICLE_IDS, f"no article under {RESULTS_DIR} has {TRACK_DIR}/{GRAPH_NAME}"
ARTICLE_ID = ARTICLE_IDS[0]          # the one section 1 walks through

print(f"{len(ARTICLE_IDS)} article(s): {ARTICLE_IDS}")
print(f"walkthrough {ARTICLE_ID} | judge {BACKEND} | batch {JUDGE_BATCH} | tag {RUN_TAG}")

20 article(s): ['art1', 'art2', 'art3', 'art4', 'art5', 'art8', 'art12', 'art15', 'art17', 'art23', 'art44', 'art46', 'art52', 'art55', 'art57', 'art58', 'art59', 'art72', 'art73', 'art74']
walkthrough art1 | judge gpt-oss-120b | batch 60 | tag main_val_r1


In [3]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None
LAST_USAGE = None

_RETRY_STATUS = (429, 500, 502, 503, 504)
_RETRY_WORDS = ("UNAVAILABLE", "RESOURCE_EXHAUSTED", "INTERNAL", "high demand", "overloaded")


# A 429 is retryable when it means "too fast" and pointless when it means "out of money": backing
# off five times against a depleted balance just burns three minutes before failing anyway.
_FATAL_WORDS = ("credits are depleted", "billing", "quota exceeded for quota metric",
                "exceeded your current quota", "insufficient")


def _gemini_retryable(exc) -> bool:
    """A transient server-side failure, as opposed to one that repeating cannot fix."""
    msg = str(exc).lower()
    if any(w in msg for w in _FATAL_WORDS):
        return False
    code = getattr(exc, "code", None) or getattr(exc, "status_code", None)
    if code in _RETRY_STATUS:
        return True
    return any(w.lower() in msg for w in _RETRY_WORDS)


def _gemini_call_with_retry(fn, attempts=5, base=4):
    """Call `fn()`, retrying TRANSIENT API errors with backoff (4s, 8s, 16s, 32s).

    Without this one `503 UNAVAILABLE - This model is currently experiencing high demand` ends the
    whole stage: it killed nb4 two articles into twenty and took nb6b and nb7b down with it. Google's
    own message says such spikes are temporary, which is what a backoff is for. A 400 or 403 still
    fails at once — repeating a malformed or unauthorised request achieves nothing.
    """
    import time
    for attempt in range(attempts):
        try:
            return fn()
        except Exception as exc:                       # noqa: BLE001 - re-raised below
            if attempt == attempts - 1 or not _gemini_retryable(exc):
                raise
            wait = min(60, base * 2 ** attempt)
            code = getattr(exc, "code", None) or getattr(exc, "status_code", "?")
            print(f"    [retry {attempt + 1}/{attempts - 1}] {type(exc).__name__} {code} — "
                  f"waiting {wait}s", flush=True)
            time.sleep(wait)


def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object, recording tokens + latency."""
    global last_latency_s, LAST_USAGE
    t0 = time.perf_counter()
    resp = _gemini_call_with_retry(lambda: client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    ))
    last_latency_s = time.perf_counter() - t0
    LAST_USAGE = nb_parallel.record_gemini_usage(resp, last_latency_s)   # prompt/output/THINKING tokens + latency
    return json.loads(resp.text)

# --- select the backend, then hand it to the shared judge ------------------------------------
# Must stay LAST in this cell: the Gemini `gemini_json` above is redefined every time the cell runs,
# so whichever binding happens last wins. `results=` keeps judge and judged independent.
import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend, grounding_core as gc
for _m in (nb_parallel, llm_backend, gc):
    importlib.reload(_m)
nb_parallel.install()

llm_backend.activate(BACKEND, ROOT, globals(), results=RESULTS_MODEL)

gc.set_llm(gemini_json)              # the judge calls whatever the line above just bound
gc.configure(judge_batch=JUDGE_BATCH, judge_retries=JUDGE_RETRIES,
             judge_retry_backoff=JUDGE_RETRY_BACKOFF)
print(f"grounding_core: batch={gc.JUDGE_BATCH} prompt={len(gc.GROUNDING_SYSTEM_PROMPT)} chars")

Note: you may need to restart the kernel to use updated packages.
backend: HOSTED https://api.fireworks.ai/inference/v1 model=accounts/fireworks/models/gpt-oss-120b thinking=True structured=True max_tokens=80000 temperature=caller top_p=None seed=None timeout=3600s params={'reasoning_effort': 'medium'}  ->  results/llama33-70b/   (reading/writing another model's tree)
grounding_core: batch=60 prompt=3112 chars


## 1. One article, walked through

Sections 1–3 ground a single article so you can read what the judge did before spending a run on the
whole split. Nothing later depends on them.

In [4]:
import json
from collections import Counter

def load_graph(article_id):
    path = RESULTS_DIR / article_id / TRACK_DIR / GRAPH_NAME
    return json.loads(path.read_text()), path

def load_article(article_id):
    return (ARTICLE_DIR / f"{article_id}.txt").read_text()

graph, graph_path = load_graph(ARTICLE_ID)
article_text = load_article(ARTICLE_ID)
art_norm, art_idx, art_raw = gc.set_article(article_text)

print(f"{graph_path.relative_to(ROOT)}")
print(f"  entities   {len(graph.get('entities', []))}")
print(f"  nary_nodes {len(graph.get('nary_nodes', []))}")
print(f"  relations  {len(graph.get('relations', []))}")
print(f"  article    {len(article_text):,} chars -> {len(art_norm):,} normalised")

results/llama33-70b/art1/two_staged/3_nary_graph/nary_graph_dedup_main_val_r1_extended_consolidated.json
  entities   29
  nary_nodes 11
  relations  36
  article    3,918 chars -> 3,917 normalised


## 2. Citation check — are the *extractor's* quotes real? (deterministic, free, non-gating)

This scores the provenance the extractor emitted; it decides nothing about what the judge sees, which
is the whole article regardless. Keeping it as its own axis is what separates *"the graph is wrong"*
from *"the graph is right but cited the wrong line"* — a distinction that disappears if the evidence
is also the judge's input.

In [5]:
graph = gc.evidence_pass(graph, art_norm, art_idx, art_raw)

for kind in ("nary_nodes", "relations"):
    c = Counter(x["grounding"]["evidence_status"] for x in graph.get(kind, []))
    print(f"{kind:11} " + "  ".join(f"{k}={c[k]}" for k in ("found", "partial", "missing", "none") if c[k]))

bad = [(x.get("id") or f'{x.get("subject_text")} -{x.get("property")}-> {x.get("object_text")}',
        x["grounding"]["evidence_detail"])
       for kind in ("nary_nodes", "relations") for x in graph.get(kind, [])
       if x["grounding"]["evidence_status"] == "missing"]
print(f"\n{len(bad)} item(s) whose cited quote is not in the article (judged anyway):")
for label, d in bad[:10]:
    print("  -", str(label)[:80], "|", d)

nary_nodes  found=11
relations   found=36

0 item(s) whose cited quote is not in the article (judged anyway):


## 3. The judge

Prompt and verbalisers come from `grounding_core`. The prompt now states explicitly that an assertion's
**property and types are part of the claim** — both endpoints appearing in the article is not enough.
That clause exists because the benchmark showed the judge passing `Person --[drugClass]--> Broadcaster`
on an article that only says she is a broadcaster.

In [6]:
print(gc.GROUNDING_SYSTEM_PROMPT[:1200], "\n   [...]\n")
_v = gc.make_relation_verbaliser(graph)
print("node    :", gc.verbalise_node(graph["nary_nodes"][0]))
print("relation:", _v(graph["relations"][0]))
_n = lambda n: (n + gc.JUDGE_BATCH - 1) // gc.JUDGE_BATCH
print(f"\nthis article: {len(graph.get('nary_nodes', []))} nodes + {len(graph.get('relations', []))} relations"
      f" -> {_n(len(graph.get('nary_nodes', [])))} + {_n(len(graph.get('relations', [])))} calls")

You are a strict grounding judge for a knowledge-graph extraction pipeline. You will be given the full text of a news article, followed by a batch of assertions extracted from it.

For each assertion, decide whether the article supports it. Judge only against the article text shown. Do not use outside knowledge, and do not assume anything the article does not state, even if you believe it is true.

A relation is written as: "SUBJECT" --[property]--> "OBJECT". Check two things:

1. The endpoints. Does the article support the subject and the object as described?

2. The property. Read the property name as plain English and ask whether the article states that specific relation between those two things. It is NOT enough that both endpoints appear in the article, and NOT enough that the article connects them in some way. The article may relate the same two things in several ways; the relation this property names must be one of them. If the property reverses the direction the article gives, 

In [7]:
# Nodes first, then the relations their nodes did not orphan (see grounding_core.ground_graph).
graph = gc.ground_graph(graph, article_text)
report = gc.grounding_report(graph)
print(json.dumps(report, indent=2))

for kind in ("nary_nodes", "relations"):
    for x in graph.get(kind, []):
        if not x["grounding"].get("grounded"):
            label = x.get("id") or f'{x.get("subject_text")} -{x.get("property")}-> {x.get("object_text")}'
            print(f'  - [{kind[:4]}] {str(label)[:60]:62} {x["grounding"].get("failure_type")}: '
                  f'{str(x["grounding"].get("note"))[:70]}')

  nodes: judged 11/11
  relations: judged 36/36
{
  "nary_nodes": {
    "total": 11,
    "grounded": 11,
    "rate": 1.0,
    "failures": {},
    "citation_status": {
      "found": 11
    },
    "citation_precision": {
      "n": 11,
      "matched": 11,
      "rate": 1.0
    },
    "judge_quote_status": {
      "found": 11
    }
  },
  "relations": {
    "total": 36,
    "grounded": 16,
    "rate": 0.444,
    "failures": {
      "intrinsic": 2,
      "extrinsic": 18
    },
    "citation_status": {
      "found": 36
    },
    "citation_precision": {
      "n": 16,
      "matched": 15,
      "rate": 0.938
    },
    "judge_quote_status": {
      "found": 16
    }
  },
  "relations_by_property_source": {
    "vocab": {
      "total": 10,
      "grounded": 10,
      "rate": 1.0
    },
    "schema.org": {
      "total": 26,
      "grounded": 6,
      "rate": 0.231
    }
  },
  "nodes_by_type_source": {
    "not_canonicalized": {
      "total": 11,
      "grounded": 11,
      "rate": 1.0


## 4. Batch — ground every article in the split

`WORKERS` articles at a time. Each article's output is buffered and printed as one block when it
finishes; a failure is reported and skipped rather than ending the run.

In [8]:
import csv

JUDGMENTS_JSON = "grounding_fulltext_judgments.json"
JUDGMENTS_CSV  = "grounding_fulltext_judgments.csv"

def write_judgment_log(graph, article_id):
    rows = gc.build_judgment_log(graph, article_id)
    d = RESULTS_DIR / article_id / TRACK_DIR
    (d / JUDGMENTS_JSON).write_text(json.dumps(rows, ensure_ascii=False, indent=2))
    with (d / JUDGMENTS_CSV).open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=gc.JUDGMENT_COLS); w.writeheader()
        for row in rows:
            flat = dict(row)
            # both may be several spans; CSV needs one cell, JSON keeps the list
            flat["evidence_cited"] = " || ".join(row["evidence_cited"] or [])
            flat["judge_quote"] = " || ".join(gc.as_quotes(row["judge_quote"]))
            w.writerow(flat)
    return rows

def ground_article(article_id):
    """Judge one article, write the graph + the judgment log, and prune. -> (report, prune report)."""
    graph, _ = load_graph(article_id)
    graph = gc.ground_graph(graph, load_article(article_id), log=lambda *a, **k: None)
    graph["grounding_report"] = gc.grounding_report(graph)
    graph["grounding_config"] = {"method": "full_article", "backend": BACKEND, "model": GEMINI_MODEL,
                                 "judge_batch": gc.JUDGE_BATCH, "graph": GRAPH_NAME,
                                 "prompt_sha1": PROMPT_SHA}
    d = RESULTS_DIR / article_id / TRACK_DIR
    (d / OUT_NAME).write_text(json.dumps(graph, ensure_ascii=False, indent=2))
    write_judgment_log(graph, article_id)

    pruned, prep = gc.prune_graph(graph, drop_unjudged=DROP_UNJUDGED,
                                  drop_orphan_entities=DROP_ORPHAN_ENTITIES)
    (d / PRUNED_NAME).write_text(json.dumps(pruned, ensure_ascii=False, indent=2))
    return graph["grounding_report"], prep

import hashlib
PROMPT_SHA = hashlib.sha1(gc.GROUNDING_SYSTEM_PROMPT.encode()).hexdigest()[:12]

def ground_and_report(aid):
    rep, prep = ground_article(aid)
    nd, rl = rep.get("nary_nodes", {}), rep.get("relations", {})
    print(f"nodes {nd.get('grounded')}/{nd.get('total')} ({nd.get('rate')})   "
          f"relations {rl.get('grounded')}/{rl.get('total')} ({rl.get('rate')})   "
          f"pruned -> {prep['nary_nodes']['after']} nodes / {prep['relations']['after']} relations")
    return rep, prep

print(f"grounding {len(ARTICLE_IDS)} article(s) with {BACKEND}, prompt {PROMPT_SHA}\n")
nb_parallel.reset_usage()
_res = nb_parallel.pmap(ground_and_report, ARTICLE_IDS, workers=WORKERS)
REPORTS = {r.item: r.value[0] for r in _res if r.ok}
PRUNES  = {r.item: r.value[1] for r in _res if r.ok}

nb_parallel.usage_report(f"nb9b · grounding ({BACKEND})", wall_seconds=nb_parallel.LAST_WALL_SECONDS,
                         write=GLOBAL_DIR / f"token_usage_grounding_{BACKEND}_{RUN_TAG}.json")

grounding 20 article(s) with gpt-oss-120b, prompt 67305d45abb6

20 item(s), running on 4 threads

[1/20] art2  (17.7s)
   nodes 3/4 (0.75)   relations 7/16 (0.438)   pruned -> 3 nodes / 7 relations
[2/20] art3  (23.4s)
   nodes 5/5 (1.0)   relations 5/11 (0.455)   pruned -> 5 nodes / 5 relations
[3/20] art4  (23.5s)
   nodes 4/6 (0.667)   relations 13/23 (0.565)   pruned -> 4 nodes / 13 relations
[4/20] art1  (36.8s)
   nodes 11/11 (1.0)   relations 16/36 (0.444)   pruned -> 11 nodes / 16 relations
[5/20] art12  (13.8s)
   nodes 5/5 (1.0)   relations 7/13 (0.538)   pruned -> 5 nodes / 7 relations
[6/20] art5  (28.5s)
   nodes 6/6 (1.0)   relations 17/23 (0.739)   pruned -> 6 nodes / 17 relations
[7/20] art8  (26.6s)
   nodes 8/8 (1.0)   relations 17/24 (0.708)   pruned -> 8 nodes / 17 relations
[8/20] art17  (17.3s)
   nodes 6/6 (1.0)   relations 15/20 (0.75)   pruned -> 6 nodes / 15 relations
[9/20] art15  (28.8s)
   nodes 9/9 (1.0)   relations 13/23 (0.565)   pruned -> 9 nodes / 13 r

{'label': 'nb9b · grounding (gpt-oss-120b)',
 'totals': {'calls': 40,
  'prompt_tokens': 86260,
  'output_tokens': 100371,
  'thoughts_tokens': 73227,
  'total_tokens': 186631,
  'seconds': 479.5},
 'wall_seconds': 124.6086990419999,
 'per_tag': {'art2': {'calls': 2,
   'seconds': 17.6,
   'prompt_tokens': 4768,
   'output_tokens': 3064,
   'thoughts_tokens': 2244,
   'total_tokens': 7832},
  'art1': {'calls': 2,
   'seconds': 36.7,
   'prompt_tokens': 4609,
   'output_tokens': 8192,
   'thoughts_tokens': 5841,
   'total_tokens': 12801},
  'art4': {'calls': 2,
   'seconds': 23.5,
   'prompt_tokens': 3062,
   'output_tokens': 4863,
   'thoughts_tokens': 3563,
   'total_tokens': 7925},
  'art3': {'calls': 2,
   'seconds': 22.9,
   'prompt_tokens': 6458,
   'output_tokens': 3743,
   'thoughts_tokens': 2834,
   'total_tokens': 10201},
  'art5': {'calls': 2,
   'seconds': 28.3,
   'prompt_tokens': 4254,
   'output_tokens': 5323,
   'thoughts_tokens': 3909,
   'total_tokens': 9577},
  'art12

## 5. Corpus-level rates

A single article's graph carries ~30 assertions, so its rate swings on two or three decisions. The
number that belongs in the thesis is the pooled one: total grounded / total asserted (micro-average —
each assertion counts once, which is what *"how much of what this pipeline emits is supported?"* asks).

Node and relation rates are reported separately and **must not be merged**: *"a node of type X exists"*
is close to tautological once the article mentions it.

In [9]:
def aggregate_reports(reports=None):
    reports = REPORTS if reports is None else reports
    agg, fails, by_prop, by_type = {}, Counter(), {}, {}
    for rep in reports.values():
        for kind in ("nary_nodes", "relations"):
            r = rep.get(kind)
            if not r:
                continue
            d = agg.setdefault(kind, {"total": 0, "grounded": 0, "cite_n": 0, "cite_matched": 0})
            d["total"] += r["total"]; d["grounded"] += r["grounded"]
            cp = r.get("citation_precision") or {}
            d["cite_n"] += cp.get("n", 0); d["cite_matched"] += cp.get("matched", 0)
            fails.update({f"{kind}:{k}": v for k, v in (r.get("failures") or {}).items() if k})
        for key, dst in (("relations_by_property_source", by_prop), ("nodes_by_type_source", by_type)):
            for src, d in (rep.get(key) or {}).items():
                t = dst.setdefault(src, {"total": 0, "grounded": 0})
                t["total"] += d["total"]; t["grounded"] += d["grounded"]
    for d in agg.values():
        d["rate"] = round(d["grounded"] / d["total"], 3) if d["total"] else None
        d["citation_precision"] = round(d["cite_matched"] / d["cite_n"], 3) if d["cite_n"] else None
    for d in [*by_prop.values(), *by_type.values()]:
        d["rate"] = round(d["grounded"] / d["total"], 3) if d["total"] else None
    return {"articles": len(reports), "article_ids": sorted(reports, key=_natkey),
            **agg, "failures": dict(fails.most_common()),
            "relations_by_property_source": by_prop, "nodes_by_type_source": by_type}

AGG = aggregate_reports()
PRUNE_AGG = {k: {"before": sum(p[k]["before"] for p in PRUNES.values()),
                 "after":  sum(p[k]["after"]  for p in PRUNES.values())}
             for k in ("nary_nodes", "relations", "entities")}
for k, d in PRUNE_AGG.items():
    d["kept_rate"] = round(d["after"] / d["before"], 3) if d["before"] else None

SUMMARY = {"backend": BACKEND, "results_model": RESULTS_MODEL, "graph": GRAPH_NAME,
           "run_tag": RUN_TAG, "prompt_sha1": PROMPT_SHA, "grounding": AGG, "pruning": PRUNE_AGG}
(GLOBAL_DIR / f"grounding_summary_{BACKEND}_{RUN_TAG}.json").write_text(json.dumps(SUMMARY, indent=2))

print(json.dumps({k: v for k, v in AGG.items() if k != "article_ids"}, indent=2))
print("\npruning:", json.dumps(PRUNE_AGG, indent=1))
print("\n! the rate above is a CEILING — 9d measures how much this judge misses")

{
  "articles": 20,
  "nary_nodes": {
    "total": 126,
    "grounded": 118,
    "cite_n": 118,
    "cite_matched": 100,
    "rate": 0.937,
    "citation_precision": 0.847
  },
  "relations": {
    "total": 419,
    "grounded": 270,
    "cite_n": 270,
    "cite_matched": 220,
    "rate": 0.644,
    "citation_precision": 0.815
  },
  "failures": {
    "relations:extrinsic": 112,
    "relations:intrinsic": 30,
    "nary_nodes:extrinsic": 8,
    "relations:orphan": 7
  },
  "relations_by_property_source": {
    "vocab": {
      "total": 224,
      "grounded": 156,
      "rate": 0.696
    },
    "schema.org": {
      "total": 195,
      "grounded": 114,
      "rate": 0.585
    }
  },
  "nodes_by_type_source": {
    "not_canonicalized": {
      "total": 126,
      "grounded": 118,
      "rate": 0.937
    }
  }
}

pruning: {
 "nary_nodes": {
  "before": 126,
  "after": 118,
  "kept_rate": 0.937
 },
 "relations": {
  "before": 419,
  "after": 270,
  "kept_rate": 0.644
 },
 "entities": {
  "be

## 6. The pruned graphs — what nb10 / MINE should read

`nary_graph_grounded_pruned.json` was written per article in section 4. Everything the judge rejected
is gone: ungrounded nodes, ungrounded relations, relations orphaned by a dropped hub, and entities no
surviving relation mentions.

The cell below is a check, not a step — it re-reads what was written and confirms nothing ungrounded
or dangling survived, because a pruning bug would be invisible downstream and would quietly inflate
whatever MINE reports.

In [10]:
bad = []
for aid in ARTICLE_IDS:
    p = RESULTS_DIR / aid / TRACK_DIR / PRUNED_NAME
    if not p.exists():
        bad.append((aid, "missing")); continue
    g = json.loads(p.read_text())
    node_ids = {n["id"] for n in g.get("nary_nodes", [])}
    ent_ids  = {e.get("id") for e in g.get("entities", [])}
    for x in g.get("nary_nodes", []) + g.get("relations", []):
        gr = x.get("grounding", {})
        if not (gr.get("grounded") or (gr.get("supported") is None and not DROP_UNJUDGED)):
            bad.append((aid, "ungrounded survived")); break
    for r in g.get("relations", []):
        for side in ("subject_id", "object_id"):
            i = r.get(side)
            if i and i not in node_ids and i not in ent_ids:
                bad.append((aid, f"dangling {side}={i}")); break

print("checked", len(ARTICLE_IDS), "pruned graphs")
print("problems:", bad if bad else "none — every remaining assertion was judged supported")
print(f"\nnb10 should read: {TRACK_DIR}/{PRUNED_NAME}")

checked 20 pruned graphs
problems: none — every remaining assertion was judged supported

nb10 should read: two_staged/3_nary_graph/nary_graph_grounded_pruned.json


## 7. Compare with nb9 (the evidence-bundle judge) — optional

nb9 shows the judge a per-assertion evidence bundle it assembles itself; a `missing` quote there skips
the assertion, so a bad citation and an unsupported claim collapse into one outcome. This notebook
shows the whole article, so it reads **higher**. That is a definitional change, not a quality gain:
the extra passes are the population nb9 was rejecting for citing a sentence one line off.

In [11]:
def compare_with_nb9(article_ids=None, examples=6):
    ids = list(article_ids or ARTICLE_IDS)
    tot, shown_up, shown_down = Counter(), [], []
    print(f"{'article':10} {'both':>6} {'nb9':>10} {'nb9b':>10} {'+':>5} {'-':>5}")
    for aid in ids:
        d = RESULTS_DIR / aid / TRACK_DIR
        p9, p9b = d / "grounding_judgments.json", d / JUDGMENTS_JSON
        if not p9.exists() or not p9b.exists():
            print(f"{aid:10} skipped — {'nb9' if not p9.exists() else 'nb9b'} log missing"); continue
        a = {r["id"]: r for r in json.loads(p9.read_text())}
        b = {r["id"]: r for r in json.loads(p9b.read_text())}
        both = [i for i in b if i in a]
        up   = [i for i in both if not a[i]["grounded"] and b[i]["grounded"]]
        down = [i for i in both if a[i]["grounded"] and not b[i]["grounded"]]
        tot.update({"both": len(both), "nb9": sum(a[i]["grounded"] for i in both),
                    "nb9b": sum(b[i]["grounded"] for i in both),
                    "up": len(up), "down": len(down), "articles": 1})
        print(f"{aid:10} {len(both):>6} {tot['nb9']:>10} {tot['nb9b']:>10} {len(up):>5} {len(down):>5}")
        shown_up += [(aid, a[i], b[i]) for i in up]; shown_down += [(aid, b[i]) for i in down]
    if not tot["articles"]:
        print("nothing to compare — run 9_grounding.ipynb over the same articles first"); return
    n = tot["both"]
    print(f"\npooled over {tot['articles']} article(s): {n} assertions in both logs")
    print(f"  nb9  grounded {tot['nb9']}/{n} ({tot['nb9']/n:.1%})")
    print(f"  nb9b grounded {tot['nb9b']}/{n} ({tot['nb9b']/n:.1%})")
    for aid, ra, rb in shown_up[:examples]:
        print(f"\n  + [{aid}] {rb['assertion'][:84]}")
        print(f"      nb9  : {ra['failure_type']}: {str(ra['justification'])[:90]}")
        print(f"      nb9b : grounded on {str(rb['judge_quote'])[:90]}")

compare_with_nb9()

article      both        nb9       nb9b     +     -
art1       skipped — nb9 log missing
art2       skipped — nb9 log missing
art3       skipped — nb9 log missing
art4       skipped — nb9 log missing
art5       skipped — nb9 log missing
art8       skipped — nb9 log missing
art12      skipped — nb9 log missing
art15      skipped — nb9 log missing
art17      skipped — nb9 log missing
art23      skipped — nb9 log missing
art44      skipped — nb9 log missing
art46      skipped — nb9 log missing
art52      skipped — nb9 log missing
art55      skipped — nb9 log missing
art57      skipped — nb9 log missing
art58      skipped — nb9 log missing
art59      skipped — nb9 log missing
art72      skipped — nb9 log missing
art73      skipped — nb9 log missing
art74      skipped — nb9 log missing
nothing to compare — run 9_grounding.ipynb over the same articles first


## 8. Every arm at once — all models x both tracks

Sections 1-7 ground one `(RESULTS_MODEL, TRACK_DIR)` at a time. This runs the whole grid with **one
judge**, which is what makes the arms comparable: a grounding rate is only a property of the graph if
the thing measuring it is held fixed.

**One arm is self-judged.** With `BACKEND = "qwen3-32b-think"`, that arm's own graphs are being
scored by the model that wrote them, and a model asked to check its own work grades generously. The
fix is not to swap judges per arm — different judges make the arms incomparable, which is worse — so
the run marks it instead: every graph gets `grounding_config.self_judged`, and the summary prints it.
If you want to bound the effect, ground that arm a second time with a different judge and compare.

**Read the rate as a ceiling.** On the 500-item benchmark this judge catches 12 of 22 real extractor
errors, so roughly 45% of what is actually wrong survives. For a graph that is ~9% wrong, expect
~4% of the assertions in the pruned output to be unsupported anyway.

Resumable: an article whose judged graph already exists is skipped, so a dropped tunnel costs only
the articles in flight. Set `FORCE = True` to redo everything.

In [12]:
# --- 8. Ground every (model, track) arm ------------------------------------------------------
ARMS = [(m, t) for m in ("gemini-2.5-flash", "llama33-70b", "qwen3-32b-think")
                for t in ("e2e", "two_staged/3_nary_graph")]
ARM_WORKERS = 8          # concurrent requests to the ONE judge server
FORCE = False            # True = re-ground articles that already have OUT_NAME

# Self-contained: sections 1 and 4 define these, and you skip both when running the grid.
import hashlib, csv, json
PROMPT_SHA = hashlib.sha1(gc.GROUNDING_SYSTEM_PROMPT.encode()).hexdigest()[:12]
JUDGMENTS_JSON = "grounding_fulltext_judgments.json"
JUDGMENTS_CSV  = "grounding_fulltext_judgments.csv"
if "load_article" not in globals():
    def load_article(aid): return (ARTICLE_DIR / f"{aid}.txt").read_text()

# Check the prerequisites BEFORE queueing 120 groundings. A missing name surfaces inside a worker,
# where it is caught per-task and reported as a failure — so the run "completes" having done nothing,
# or worse, having spent the API calls first and crashed on the write.
_need = ["ROOT", "ARTICLE_DIR", "BACKEND", "GEMINI_MODEL", "GRAPH_NAME", "OUT_NAME", "PRUNED_NAME",
         "RUN_TAG", "DROP_UNJUDGED", "DROP_ORPHAN_ENTITIES", "_natkey", "gc", "nb_parallel"]
_absent = [n for n in _need if n not in globals()]
assert not _absent, f"run the config and client cells first — missing: {_absent}"
assert gc._LLM is not None, "no judge bound — re-run the client cell (it calls gc.set_llm)"

def _arm_dir(model, track, aid):
    return ROOT / "results" / model / aid / track

def ground_one(task):
    model, track, aid = task
    d = _arm_dir(model, track, aid)
    graph = json.loads((d / GRAPH_NAME).read_text())
    graph = gc.ground_graph(graph, load_article(aid), log=lambda *a, **k: None)
    graph["grounding_report"] = gc.grounding_report(graph)
    graph["grounding_config"] = {"method": "full_article", "judge_backend": BACKEND,
                                 "judge_model": GEMINI_MODEL, "judge_batch": gc.JUDGE_BATCH,
                                 "prompt_sha1": PROMPT_SHA, "graph": GRAPH_NAME,
                                 "self_judged": model == BACKEND}
    (d / OUT_NAME).write_text(json.dumps(graph, ensure_ascii=False, indent=2))

    rows = gc.build_judgment_log(graph, aid)
    (d / JUDGMENTS_JSON).write_text(json.dumps(rows, ensure_ascii=False, indent=2))
    with (d / JUDGMENTS_CSV).open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=gc.JUDGMENT_COLS); w.writeheader()
        for row in rows:
            flat = dict(row)
            flat["evidence_cited"] = " || ".join(row["evidence_cited"] or [])
            flat["judge_quote"] = " || ".join(gc.as_quotes(row["judge_quote"]))
            w.writerow(flat)

    pruned, prep = gc.prune_graph(graph, drop_unjudged=DROP_UNJUDGED,
                                  drop_orphan_entities=DROP_ORPHAN_ENTITIES)
    (d / PRUNED_NAME).write_text(json.dumps(pruned, ensure_ascii=False, indent=2))
    return {"arm": (model, track), "aid": aid,
            "report": graph["grounding_report"], "prune": prep}

# Interleave the arms so a run that dies early leaves SOME of every arm rather than all of one.
_tasks, _skipped = [], 0
_per_arm = []
for _m, _t in ARMS:
    ids = [a.name for a in (ROOT / "results" / _m).iterdir()
           if a.is_dir() and (a / _t / GRAPH_NAME).exists()]
    _per_arm.append([(_m, _t, a) for a in sorted(ids, key=_natkey)])
from itertools import zip_longest
for _row in zip_longest(*_per_arm):
    for _task in _row:
        if _task is None:
            continue
        # PRUNED_NAME is written LAST, so it is the only safe "this article is done" marker:
        # OUT_NAME exists as soon as the judged graph lands, and an article that died after that
        # would be skipped forever with no pruned graph and no judgment log.
        if not FORCE and (_arm_dir(*_task) / PRUNED_NAME).exists():
            _skipped += 1; continue
        _tasks.append(_task)

print(f"{sum(len(x) for x in _per_arm)} article-groundings over {len(ARMS)} arms; "
      f"{_skipped} already done, {len(_tasks)} to run")
print(f"judge {BACKEND} (prompt {PROMPT_SHA}), batch {gc.JUDGE_BATCH}, {ARM_WORKERS} workers")
_self = [f"{m}/{t.split('/')[0]}" for m, t in ARMS if m == BACKEND]
if _self:
    print(f"!! self-judged arm(s): {_self} — flagged in every graph's grounding_config")

if _tasks:
    nb_parallel.reset_usage()
    _res = nb_parallel.pmap(ground_one, _tasks, workers=ARM_WORKERS,
                            label=lambda t: f"{t[0][:12]}/{t[1].split('/')[0][:4]}/{t[2]}")
    _bad = [r.item for r in _res if not r.ok]
    if _bad:
        print(f"\n!! {len(_bad)} failed: {_bad[:6]} — re-run this cell to retry only those")
    nb_parallel.usage_report(f"nb9b · all arms ({BACKEND})",
                             wall_seconds=nb_parallel.LAST_WALL_SECONDS,
                             write=ROOT / "results" / f"token_usage_grounding_all_arms_{RUN_TAG}.json")

120 article-groundings over 6 arms; 120 already done, 0 to run
judge gpt-oss-120b (prompt 67305d45abb6), batch 60, 8 workers


In [13]:
# --- 8b. Per-arm summary, written to each model's _global/ ----------------------------------
from collections import Counter

def summarise_arm(model, track):
    ids = sorted([a.name for a in (ROOT / "results" / model).iterdir()
                  if a.is_dir() and (a / track / OUT_NAME).exists()], key=_natkey)
    agg, pr = {}, Counter()
    for aid in ids:
        g = json.loads((_arm_dir(model, track, aid) / OUT_NAME).read_text())
        rep = g.get("grounding_report", {})
        for kind in ("nary_nodes", "relations"):
            r = rep.get(kind)
            if not r: continue
            d = agg.setdefault(kind, {"total": 0, "grounded": 0})
            d["total"] += r["total"]; d["grounded"] += r["grounded"]
        p = json.loads((_arm_dir(model, track, aid) / PRUNED_NAME).read_text()).get("prune_report", {})
        for k in ("nary_nodes", "relations", "entities"):
            pr[k + "_before"] += p.get(k, {}).get("before", 0)
            pr[k + "_after"] += p.get(k, {}).get("after", 0)
    for d in agg.values():
        d["rate"] = round(d["grounded"] / d["total"], 3) if d["total"] else None
    return {"model": model, "track": track, "articles": len(ids), "grounding": agg,
            "pruning": {k: {"before": pr[k + "_before"], "after": pr[k + "_after"],
                            "kept_rate": round(pr[k + "_after"] / pr[k + "_before"], 3)
                                         if pr[k + "_before"] else None}
                        for k in ("nary_nodes", "relations", "entities")},
            "judge_backend": BACKEND, "prompt_sha1": PROMPT_SHA,
            "self_judged": model == BACKEND, "run_tag": RUN_TAG}

SUMMARIES = {}
print(f"{'arm':40} {'arts':>5} {'nodes':>13} {'relations':>15} {'kept':>7}")
for _m, _t in ARMS:
    s = summarise_arm(_m, _t)
    SUMMARIES[(_m, _t)] = s
    out = ROOT / "results" / _m / "_global" / f"grounding_summary_{BACKEND}_{_t.split('/')[0]}_{RUN_TAG}.json"
    out.write_text(json.dumps(s, indent=2))
    nd, rl = s["grounding"].get("nary_nodes", {}), s["grounding"].get("relations", {})
    kept = s["pruning"]["relations"]["kept_rate"]
    flag = "  <- self-judged" if s["self_judged"] else ""
    print(f"{_m + ' / ' + _t.split('/')[0]:40} {s['articles']:>5} "
          f"{nd.get('grounded',0):>5}/{nd.get('total',0):<5} ({nd.get('rate')})"
          f" {rl.get('grounded',0):>6}/{rl.get('total',0):<6} ({rl.get('rate')}) {kept:>6}{flag}")

print("\n! these rates are CEILINGS — the judge misses ~45% of real extractor errors (see 9d)")
print(f"  nb10 / MINE should read: <model>/<article>/<track>/{PRUNED_NAME}")

arm                                       arts         nodes       relations    kept
gemini-2.5-flash / e2e                      99  2672/2696  (0.991)  11382/13021  (0.874)  0.879
gemini-2.5-flash / two_staged               20   732/752   (0.973)   2717/3299   (0.824)  0.824
llama33-70b / e2e                          100   359/378   (0.95)   1169/1801   (0.649)   0.65
llama33-70b / two_staged                    20   118/126   (0.937)    270/419    (0.644)  0.644
qwen3-32b-think / e2e                       20   154/156   (0.987)    405/533    (0.76)   0.76
qwen3-32b-think / two_staged                20   187/199   (0.94)    387/530    (0.73)   0.73

! these rates are CEILINGS — the judge misses ~45% of real extractor errors (see 9d)
  nb10 / MINE should read: <model>/<article>/<track>/nary_graph_grounded_pruned.json
